# 🔤 Assamese OCR — Sentence Model Training (Kaggle)

This notebook trains a CRNN (CNN + LSTM) model for Assamese sentence-level OCR using CTC loss.

**Before you start:**
1. Set accelerator to **GPU** → Settings (right panel) → Accelerator → GPU T4 x2
2. Add your dataset containing `as-wiki-2021.txt` as a Kaggle Dataset input
3. (Optional) Add a dataset with your existing checkpoints `.pth` files

**Kaggle paths:**
- Input datasets: `/kaggle/input/<dataset-slug>/`
- Working directory (writable): `/kaggle/working/`
- Checkpoints will be saved to `/kaggle/working/checkpoints/`

---

## 1. Setup — Clone Repo & Install Dependencies

In [ ]:
import os

REPO_URL    = "https://github.com/rishav660/Assamese_OCR.git"
REPO_NAME   = "Assamese_OCR"
BRANCH      = "main"
WORK_DIR    = f'/kaggle/working/{REPO_NAME}/django_backend'

# Ensure /kaggle/working exists and shell cwd is valid (can get lost between cells)
os.makedirs('/kaggle/working', exist_ok=True)
os.chdir('/kaggle/working')

# Remove any existing clone and reclone fresh from the selected branch
if os.path.exists(f'/kaggle/working/{REPO_NAME}'):
    print('Removing stale clone...')
    !rm -rf /kaggle/working/{REPO_NAME}

print(f'Cloning {REPO_URL} branch {BRANCH}...')
!git clone -b {BRANCH} {REPO_URL} /kaggle/working/{REPO_NAME}

# Verify clone was successful before proceeding
if os.path.exists(WORK_DIR):
    # Switch to working directory
    %cd {WORK_DIR}
    !pwd
    print('✅ Successfully cloned and switched to working directory')
else:
    print(f'❌ Clone failed - {WORK_DIR} does not exist')
    print('Please check the git clone output above for errors')

In [ ]:
# Install dependencies
!pip install -q -r requirements-train.txt

In [ ]:
# Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected. Enable GPU in Settings → Accelerator.')

## 2. Locate Input Data & Set Up Directories

Your `as-wiki-2021.txt` must be added as a Kaggle Dataset.

**How to add it:**
1. Upload `as-wiki-2021.txt` to a Kaggle Dataset (or use an existing one)
2. In this notebook → top-right → **+ Add Data** → search for your dataset
3. It will appear at `/kaggle/input/<your-dataset-slug>/as-wiki-2021.txt`

Update `WIKI_INPUT_PATH` below to match your dataset slug.

In [ ]:
import os
import shutil

# ─── UPDATE THIS to match your Kaggle dataset slug ───────────────────────────
WIKI_INPUT_PATH = '/kaggle/input/assamese-wiki/as-wiki-2021.txt'  # ← CHANGE THIS
# ─────────────────────────────────────────────────────────────────────────────

# Verify the wiki file exists
if os.path.exists(WIKI_INPUT_PATH):
    size_mb = os.path.getsize(WIKI_INPUT_PATH) / 1e6
    print(f'✅ Wiki file found: {WIKI_INPUT_PATH} ({size_mb:.1f} MB)')
else:
    print(f'❌ Wiki file not found at: {WIKI_INPUT_PATH}')
    print('   Add your dataset via "+ Add Data" in the right panel.')

# Create local data/ and checkpoints/ directories under the working dir
os.makedirs('data', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

# Symlink wiki file into data/ for convenience
wiki_link = 'data/as-wiki-2021.txt'
if not os.path.exists(wiki_link):
    os.symlink(WIKI_INPUT_PATH, wiki_link)
    print(f'✅ Symlinked wiki file → data/as-wiki-2021.txt')
else:
    print(f'✅ data/as-wiki-2021.txt already linked')

In [ ]:
# (Optional) Copy existing checkpoints from a Kaggle dataset input
# Add your checkpoint dataset via "+ Add Data", then update the path below.

CHECKPOINT_INPUT = '/kaggle/input/assamese-ocr-checkpoints'  # ← CHANGE or comment out

if os.path.exists(CHECKPOINT_INPUT):
    for f in os.listdir(CHECKPOINT_INPUT):
        if f.endswith('.pth'):
            src = os.path.join(CHECKPOINT_INPUT, f)
            dst = os.path.join('checkpoints', f)
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
                print(f'  Copied: {f}')
            else:
                print(f'  Already exists: {f}')
    print('✅ Checkpoints ready')
else:
    print('ℹ️  No checkpoint input dataset found — training from scratch.')

print('\nCheckpoints directory:')
!ls -lh checkpoints/ 2>/dev/null || echo '  (empty)'

## 3. Generate Non-Overlapping Train / Val / Test Splits

In [ ]:
# Check if splits already exist (useful if re-running the notebook)
train_labels = 'data/train_real_sentences/labels/labels.txt'
if os.path.exists(train_labels):
    with open(train_labels) as f:
        n = sum(1 for _ in f)
    print(f'✅ Splits already exist! ({n} training samples)')
    print('Skip the next cell unless you want to regenerate.')
else:
    print('❌ No splits found. Run the next cell to generate them.')

In [ ]:
# Step 1: Clean up any old split directories
for split in ['train_real_sentences', 'val_real_sentences', 'test_real_sentences']:
    dst = f'data/{split}'
    if os.path.islink(dst):
        os.unlink(dst)
    elif os.path.exists(dst):
        shutil.rmtree(dst)
    os.makedirs(f'{dst}/images', exist_ok=True)
    os.makedirs(f'{dst}/labels', exist_ok=True)

print('✅ Local data directories ready')

# Step 2: Generate splits
!python build_real_sentence_splits.py \
    --input data/as-wiki-2021.txt \
    --train-output data/train_real_sentences \
    --val-output data/val_real_sentences \
    --test-output data/test_real_sentences \
    --train-count 50000 \
    --val-count 5000 \
    --test-count 2000 \
    --seed 42

# Step 3: Verify generated splits
print()
for split in ['train_real_sentences', 'val_real_sentences', 'test_real_sentences']:
    img_dir    = f'data/{split}/images'
    label_file = f'data/{split}/labels/labels.txt'
    n_images   = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    ok         = '✅' if os.path.exists(label_file) else '❌'
    print(f'{split}')
    print(f'  Images     : {n_images}')
    print(f'  labels.txt : {ok}')

print('\n✅ Dataset generation complete')

## 4. Train the Sentence Model 🚀

In [ ]:
# Main training run
!python train_sentence_model.py \
    --train-img-dir   data/train_real_sentences/images \
    --train-label-file data/train_real_sentences/labels/labels.txt \
    --val-img-dir     data/val_real_sentences/images \
    --val-label-file  data/val_real_sentences/labels/labels.txt \
    --epochs 25 \
    --batch-size 32 \
    --learning-rate 0.0001 \
    --best-checkpoint  checkpoints/best_model_sentences.pth \
    --final-checkpoint checkpoints/final_model_sentences.pth \
    --latest-checkpoint checkpoints/latest_model_sentences.pth \
    --resume \
    --plot-out training_curve_sentences.png

In [ ]:
# View training curve
from IPython.display import Image as IPImage, display
if os.path.exists('training_curve_sentences.png'):
    display(IPImage('training_curve_sentences.png'))
else:
    print('No training curve found yet.')

## 5. Evaluate Model Accuracy (CER / WER / Exact Match)

In [ ]:
# Evaluate on VALIDATION set — raw model output (greedy decode)
!python evaluate.py \
    --img-dir    data/val_real_sentences/images \
    --label-file data/val_real_sentences/labels/labels.txt \
    --checkpoint checkpoints/best_model_sentences.pth \
    --num-samples 10

In [ ]:
# Evaluate on VALIDATION set — with spell-check post-processing
# Compare with the cell above to see if post-processing helps
!python evaluate.py \
    --img-dir    data/val_real_sentences/images \
    --label-file data/val_real_sentences/labels/labels.txt \
    --checkpoint checkpoints/best_model_sentences.pth \
    --post-process \
    --num-samples 10

In [ ]:
# Evaluate on VALIDATION set — CTC Beam Search (typically better accuracy)
!python evaluate.py \
    --img-dir    data/val_real_sentences/images \
    --label-file data/val_real_sentences/labels/labels.txt \
    --checkpoint checkpoints/best_model_sentences.pth \
    --beam-width 10 \
    --num-samples 10

In [ ]:
# Evaluate on TEST set
test_labels = 'data/test_real_sentences/labels/labels.txt'
if os.path.exists(test_labels):
    !python evaluate.py \
        --img-dir    data/test_real_sentences/images \
        --label-file {test_labels} \
        --checkpoint checkpoints/best_model_sentences.pth \
        --num-samples 10 \
        --output test_results.tsv
    print('\n✅ Detailed results saved to test_results.tsv')
else:
    print('❌ No test set found. Regenerate splits with --test-count 2000.')

In [ ]:
# Quick single-image prediction with ground-truth comparison
!python predict_cli.py \
    --image        data/val_real_sentences/images/sentence_000000.png \
    --checkpoint   checkpoints/best_model_sentences.pth \
    --ground-truth data/val_real_sentences/labels/sentence_000000.txt

## 6. (Optional) Transfer Learning

If you have `best_model_fast.pth` in `checkpoints/`, you can fine-tune from the character model.

In [ ]:
# Transfer learning — uncomment to run
# !python train_transfer_learning.py

## 7. Save Outputs

On Kaggle, anything written to `/kaggle/working/` is automatically saved as notebook output when the session ends.  
You can download your checkpoints, results, and plots from the **Output** tab after the run completes.

In [ ]:
# List all saved outputs
print('=== Checkpoints ===')
!ls -lh checkpoints/ 2>/dev/null || echo '  (none)'

print('\n=== Plots & Results ===')
!ls -lh *.png *.tsv 2>/dev/null || echo '  (none)'

print()
print('✅ All files in /kaggle/working/ are available in the Output tab.')

In [ ]:
# (Optional) Push code changes back to GitHub
# Uncomment and fill in your credentials / token if needed

# import subprocess
# GITHUB_TOKEN = 'your_token_here'  # Use a Kaggle Secret instead
# repo_dir = f'/kaggle/working/{REPO_NAME}'
# subprocess.run(['git', '-C', repo_dir, 'add', '-A'])
# subprocess.run(['git', '-C', repo_dir, 'commit', '-m', 'Kaggle training updates'])
# subprocess.run(['git', '-C', repo_dir, 'push', 'origin', 'develop'])